# FLO CLTV — Model Geliştirme ve Segment Analizi

EDA'nın ardından bu notebook'ta:
1. BG/NBD ve Gamma-Gamma modelleri kurulur
2. 3 ve 6 aylık satın alma tahminleri üretilir
3. 6 aylık CLTV hesaplanır
4. Müşteriler segmentlere ayrılır
5. Segmentler yorumlanır ve aksiyon önerileri sunulur

In [ ]:
import sys
sys.path.insert(0, "..")

import yaml
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.data_loader import load_data, preprocess
from src.cltv import (
    build_cltv_dataframe,
    fit_bgf, fit_ggf,
    predict_purchases, predict_cltv,
    assign_segments,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")
sns.set_theme(style="whitegrid")

with open("../config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

cfg = config["analysis"]

## 1. Veri Hazırlama

In [ ]:
df_raw = load_data(config["data"]["raw_path"])
df = preprocess(df_raw, outlier_cols=config["outlier_cols"])
cltv_df = build_cltv_dataframe(df, analysis_date=cfg["analysis_date"])
print(f"Model için hazır müşteri sayısı (frequency > 1): {len(cltv_df):,}")
cltv_df.describe().T

## 2. BG/NBD Modeli — Satın Alma Tahmini

BG/NBD (Beta Geometric / Negative Binomial Distribution) modeli her müşteri için:
- Hâlâ aktif olma olasılığını
- Gelecekte kaç satın alma yapacağını

tahmin eder.

In [ ]:
bgf = fit_bgf(cltv_df, penalizer=cfg["penalizer_bgf"])
print("Model parametreleri:")
print(bgf.params_)

In [ ]:
cltv_df = predict_purchases(cltv_df, bgf, months=cfg["prediction_months"])

print("3 ayda en fazla satın alma beklenen 10 müşteri:")
cltv_df.sort_values("exp_sales_3_month", ascending=False)[
    ["customer_id", "frequency", "recency_cltv_weekly", "exp_sales_3_month"]
].head(10)

In [ ]:
print("6 ayda en fazla satın alma beklenen 10 müşteri:")
cltv_df.sort_values("exp_sales_6_month", ascending=False)[
    ["customer_id", "frequency", "recency_cltv_weekly", "exp_sales_6_month"]
].head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(cltv_df["exp_sales_3_month"], bins=50, color="steelblue",
             edgecolor="white", linewidth=0.4)
axes[0].set_title("3 Aylık Beklenen Satış Dağılımı")
axes[0].set_xlabel("Beklenen Satın Alma")
axes[0].set_ylabel("Müşteri Sayısı")

axes[1].hist(cltv_df["exp_sales_6_month"], bins=50, color="seagreen",
             edgecolor="white", linewidth=0.4)
axes[1].set_title("6 Aylık Beklenen Satış Dağılımı")
axes[1].set_xlabel("Beklenen Satın Alma")

plt.tight_layout()
plt.show()

## 3. Gamma-Gamma Modeli — Ortalama Değer Tahmini

Gamma-Gamma modeli, aktif müşterinin bir sonraki alışverişte
ne kadar harcayacağını tahmin eder.

> **Not:** Bu model frequency ile monetary arasında korelasyon olmadığını
> varsayar. Aşağıda kontrol edilmiştir.

In [ ]:
corr = cltv_df[["frequency", "monetary_cltv_avg"]].corr().iloc[0, 1]
print(f"Frequency — Monetary korelasyonu: {corr:.4f}")
print("Model varsayımı için 0'a yakın olması beklenir.")

In [ ]:
ggf = fit_ggf(cltv_df, penalizer=cfg["penalizer_ggf"])
print("Model parametreleri:")
print(ggf.params_)

In [ ]:
cltv_df = predict_cltv(
    cltv_df, bgf, ggf,
    time=6,
    discount_rate=cfg["discount_rate"],
)

print("CLTV en yüksek 20 müşteri:")
cltv_df.sort_values("cltv", ascending=False)[
    ["customer_id", "frequency", "monetary_cltv_avg",
     "exp_sales_6_month", "exp_average_value", "cltv"]
].head(20)

## 4. Segmentasyon

In [ ]:
cltv_df = assign_segments(cltv_df, n_segments=cfg["n_segments"])
cltv_df["cltv_segment"].value_counts().sort_index()

In [ ]:
segment_summary = (
    cltv_df.groupby("cltv_segment")
    .agg(
        musteri_sayisi=("customer_id", "count"),
        ort_recency=("recency_cltv_weekly", "mean"),
        ort_frequency=("frequency", "mean"),
        ort_monetary=("monetary_cltv_avg", "mean"),
        ort_exp_satis_6ay=("exp_sales_6_month", "mean"),
        ort_cltv=("cltv", "mean"),
        toplam_cltv=("cltv", "sum"),
    )
    .round(2)
)
segment_summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# CLTV dağılımı segment bazında
for segment, group in cltv_df.groupby("cltv_segment"):
    axes[0].hist(group["cltv"], bins=30, alpha=0.6, label=f"Segment {segment}")
axes[0].set_title("Segment Bazında CLTV Dağılımı")
axes[0].set_xlabel("CLTV")
axes[0].set_ylabel("Müşteri Sayısı")
axes[0].legend()

# Ortalama CLTV
axes[1].bar(segment_summary.index, segment_summary["ort_cltv"],
            color=sns.color_palette("Blues_r", 4))
axes[1].set_title("Segment Bazında Ortalama CLTV")
axes[1].set_ylabel("Ortalama CLTV (₺)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# Toplam CLTV (pasta)
axes[2].pie(
    segment_summary["toplam_cltv"],
    labels=segment_summary.index,
    autopct="%1.1f%%",
    colors=sns.color_palette("Blues_r", 4),
    startangle=90,
)
axes[2].set_title("Toplam CLTV'de Segment Payları")

plt.tight_layout()
plt.show()

## 5. Segment Yorumları ve Aksiyon Önerileri

### Segment A — En Değerli Müşteriler
Şirketin en yüksek CLTV'ye sahip müşterileri. Sık alışveriş yapıyor,
ortalama sepet değerleri yüksek.

**Önerilen aksiyonlar:**
- Özel sadakat programına dahil et
- Yeni ürün/koleksiyon lansmanlarında erken erişim sun
- Kişiselleştirilmiş iletişimle ilişkiyi güçlendir

---

### Segment B — Yüksek Potansiyelli Müşteriler
Düzenli alışveriş yapıyor ancak A segmentine henüz ulaşamamış.
Doğru teşviklerle A'ya taşınabilirler.

**Önerilen aksiyonlar:**
- Sepet değerini artırmaya yönelik cross-sell / upsell kampanyaları
- Alışveriş sıklığını artıracak hatırlatma e-postaları
- Kategori derinleştirme teklifleri

---

### Segment C — Geliştirilmesi Gereken Müşteriler
Orta düzey alışveriş sıklığı ve harcaması var.
Aktivasyon kampanyalarına iyi yanıt verebilirler.

**Önerilen aksiyonlar:**
- Belirli eşiği geçince indirim/hediye kuponu kampanyaları
- İlgilendiği kategorilerde özel teklifler

---

### Segment D — Düşük Etkileşimli Müşteriler
Yeni gelmiş ya da seyrek alışveriş yapan müşteriler.
Yüksek maliyet gerektiren aksiyonlardan kaçınılmalı.

**Önerilen aksiyonlar:**
- Düşük maliyetli e-posta / push notification dokunuşları
- İlk tekrar alışverişi teşvik eden hoş geldin kampanyası

## 6. Çıktıyı Kaydet

In [ ]:
out_path = "../" + config["data"]["processed_path"]
import os; os.makedirs(os.path.dirname(out_path), exist_ok=True)
cltv_df.to_csv(out_path, index=False)
print(f"Kaydedildi → {out_path}")
cltv_df.head()